# CSC 4110 Machine Learning: task notebook


| Field | Fill in |
|---|---|
| Task | 9.1 Model packaging and web application |
| Owner (name) | Peter Mbugua |
| Registration number | COM/0039/23 |
| Group | G04 |
| Dataset | Credit Risk Dataset |
| Date started | 26 November 2026 |


**Rules**
- Save this file in your group folder as `G<No>_U<task>_<short name>.ipynb`, exactly as shown under **Save as** in your group's paper, e.g. `G01_U2.1_cleaning.ipynb`.
- One Markdown heading per part, then its code cells. Comment every code cell.
- Use the exact variable and file names in **Graded evidence** of your task, and `random_state=42` everywhere.
- Reviewers add a cell headed **Review by <Name>** in the Reviews section. They never edit the owner's cells.

In [1]:
# Setup: set your group and task, connect Google Drive, and work inside the group folder
GROUP = "G04"        # your group, two digits
UNIT = "9.1"         # your task number: 2.1, 2.2, 3.1, 3.2, 4, 5, 6, 7, 8, 9.1, 9.2 or 9.3
RANDOM_STATE = 42    # use this everywhere so the grader gets the same numbers

import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted here:", e)   # normal when the grader runs this notebook

BASE = f"/content/drive/MyDrive/CSC4110_{GROUP}"
for sub in ["data", "notebooks", "models", "outputs", "app", "report", "tests"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)
os.chdir(BASE)   # every path below is relative to the group folder
print("Working in", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in /content/drive/.shortcut-targets-by-id/1Hfur0Lf8z0gktOkV4oQdkhdGubaALOUm/CSC4110_G04


In [2]:
# Libraries used in most tasks, and a helper that saves every chart to outputs/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def save_fig(name):
    """Save the current chart as outputs/<name>.png, then show it."""
    path = f"outputs/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    print("Saved", path)
    plt.show()

## Load the data

- Task 2.1 loads the original Kaggle file.
- Task 2.2 loads `data/stage1_<dataset>.csv`.
- Every other task loads `data/clean_<dataset>.csv`.

In [3]:

# Task 9.1: Load the Mulika SACCO Credit Risk Dataset

import os
import pandas as pd
import numpy as np

# Use the cleaned dataset from the group project
DATA_FILE = "data/clean_credit.csv"

# Check that the correct dataset exists
if not os.path.isfile(DATA_FILE):
    raise FileNotFoundError(
        "Dataset not found. Check the Group G04 folder."
    )

# Load the dataset
df = pd.read_csv(DATA_FILE)

# Display the dataset information
print("Dataset shape:", df.shape)

print("\nLoan default counts:")
print(df["loan_status"].value_counts())

# Display the first five records
display(df.head())

# Verify the target variable
print("\nClassification target:",
      "loan_status" in df.columns)

print("\nTask 9.1 dataset loaded successfully!")


Dataset shape: (32409, 25)

Loan default counts:
loan_status
0    25321
1     7088
Name: count, dtype: int64


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,...,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,age_band,log_income
0,21,9600,OWN,5.0,EDUCATION,1,1000,11.14,0,0.10,...,1,0,0,1,0,0,0,0,under 25,9.169518
1,25,9600,MORTGAGE,1.0,MEDICAL,2,5500,12.87,1,0.57,...,0,0,0,0,0,1,0,0,25-34,9.169518
2,23,65500,RENT,4.0,MEDICAL,2,35000,15.23,1,0.53,...,0,1,0,0,0,1,0,0,under 25,11.089805
3,24,54400,RENT,8.0,MEDICAL,2,35000,14.27,1,0.55,...,0,1,0,0,0,1,0,0,under 25,10.904119
4,21,9900,OWN,2.0,VENTURE,0,2500,7.14,1,0.25,...,1,0,0,0,0,0,0,1,under 25,9.200290



Classification target: True

Task 9.1 dataset loaded successfully!



## Task 9.1(a): Model Packaging

This section retrains the loan-default classifier as a complete machine learning pipeline.

The pipeline includes missing-value imputation, numerical preprocessing, categorical encoding and Random Forest classification.

I save the trained pipeline using joblib so that the Streamlit application can load it and predict loan-default probabilities for new applicants.


In [4]:

# Task 9.1(a): Train the final classification pipeline

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Select original applicant and loan features
feature_columns = [
    "person_age",
    "person_income",
    "person_home_ownership",
    "person_emp_length",
    "loan_intent",
    "loan_grade",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length"
]

# Prepare input features and target
X = df[feature_columns].copy()
y = df["loan_status"].astype(int)

# Standardise previous-default categories as Y or N
X["cb_person_default_on_file"] = (
    X["cb_person_default_on_file"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"0": "N", "1": "Y"})
)

# Define numerical columns
numeric_columns = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_grade",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

# Define categorical columns
categorical_columns = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file"
]

# Create a stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Preprocess numerical values
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical values
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

# Build the required complete pipeline
final_pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the full pipeline on training data
final_pipe.fit(X_train, y_train)

# Evaluate on held-out test data
predictions = final_pipe.predict(X_test)

print("FINAL RANDOM FOREST PIPELINE")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Original input features:", X_train.shape[1])

print("\nMODEL PERFORMANCE")
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("Precision:", round(precision_score(y_test, predictions), 4))
print("Recall:", round(recall_score(y_test, predictions), 4))
print("F1-score:", round(f1_score(y_test, predictions), 4))

print("\nPipeline trained successfully!")


FINAL RANDOM FOREST PIPELINE
Training records: 25927
Testing records: 6482
Original input features: 10

MODEL PERFORMANCE
Accuracy: 0.9367
Precision: 0.9764
Recall: 0.7285
F1-score: 0.8344

Pipeline trained successfully!


In [5]:

# Task 9.1(a): Save the complete pipeline

import os
import joblib

# Confirm that we are working in the correct group folder
print("Current folder:", os.getcwd())

assert os.path.basename(os.getcwd()) == "CSC4110_G04", (
    "Change to the Group G04 project folder first."
)

# Create the models directory if necessary
os.makedirs("models", exist_ok=True)

# Save preprocessing and trained model together
MODEL_PATH = "models/G04_model.joblib"

joblib.dump(final_pipe, MODEL_PATH)

# Verify the saved file
print("\nMODEL SAVED SUCCESSFULLY")
print("File:", MODEL_PATH)
print("File exists:", os.path.isfile(MODEL_PATH))
print("File size:", round(
    os.path.getsize(MODEL_PATH) / (1024 * 1024), 2
), "MB")


Current folder: /content/drive/.shortcut-targets-by-id/1Hfur0Lf8z0gktOkV4oQdkhdGubaALOUm/CSC4110_G04

MODEL SAVED SUCCESSFULLY
File: models/G04_model.joblib
File exists: True
File size: 39.9 MB


In [6]:

# Task 9.1(a): Reload and test the saved model

import joblib

# Reload the model saved in the previous cell
loaded_model = joblib.load("models/G04_model.joblib")

# Select one held-out test record
example = X_test.iloc[[0]]

# Predict whether the loan will default
prediction = loaded_model.predict(example)[0]

# Calculate the probability of default
default_probability = loaded_model.predict_proba(
    example
)[0, 1]

# Display the results
print("SAVED MODEL TEST")

print("Predicted loan status:", prediction)
print("Default probability:",
      round(default_probability * 100, 2), "%")

print("Actual loan status:", y_test.iloc[0])

print("\nModel reloaded and tested successfully!")


SAVED MODEL TEST
Predicted loan status: 0
Default probability: 11.0 %
Actual loan status: 0

Model reloaded and tested successfully!


In [7]:

# Compress the saved model for easier GitHub upload

import joblib
import os

MODEL_PATH = "models/G04_model.joblib"

# Save the same trained pipeline with compression
joblib.dump(final_pipe, MODEL_PATH, compress=3)

# Confirm the compressed model still works
loaded_model = joblib.load(MODEL_PATH)

assert loaded_model.predict(X_test.iloc[[0]])[0] == final_pipe.predict(X_test.iloc[[0]])[0]

size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)

print("Compressed model size:", round(size_mb, 2), "MB")
print("Saved model reloaded successfully!")


Compressed model size: 7.18 MB
Saved model reloaded successfully!



## Task 9.1(b): Building the Streamlit Web Application

I developed a Streamlit application for Mulika SACCO that accepts loan applicant details and uses the saved Random Forest pipeline to estimate loan-default probability.

The application identifies applicants who require further review by a loan officer. It supports, but does not replace, human lending decisions.


In [8]:

%%writefile app/app.py
# Mulika SACCO - Group G04 Loan Default Screening App

from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

# Resolve the project directory
ROOT = Path(__file__).resolve().parent.parent
MODEL_PATH = ROOT / "models" / "G04_model.joblib"

# Provisional threshold based on Task 5 analysis
THRESHOLD = 0.30

# Configure the Streamlit page
st.set_page_config(
    page_title="Mulika SACCO Loan Risk",
    page_icon="🏦",
    layout="centered"
)

st.title("🏦 Mulika SACCO")
st.subheader("Loan Default Risk Screening")

st.write("**Group:** G04")
st.write("**Dataset:** Credit Risk Dataset")

with st.expander("Project Team"):
    st.write(
        "Festus Ligeyo, Collins Kipkorir, "
        "Chirchir Collins, Peter Mbugua, "
        "Hartie Omuruon"
    )

st.info(
    "This application supports loan officers. "
    "Predictions are not automatic loan approval "
    "or rejection decisions."
)

# Load the saved machine learning pipeline
@st.cache_resource
def load_pipeline():
    return joblib.load(MODEL_PATH)

if not MODEL_PATH.is_file():
    st.error("Trained model file not found.")
    st.stop()

model = load_pipeline()

st.divider()
st.header("Applicant Information")

st.caption(
    "Enter amounts in the same currency and units "
    "used by the original training dataset."
)

# Collect loan application details
with st.form("loan_application"):

    col1, col2 = st.columns(2)

    with col1:
        age = st.number_input(
            "Applicant Age",
            min_value=18,
            max_value=100,
            value=30
        )

        income = st.number_input(
            "Annual Income",
            min_value=1.0,
            value=60000.0,
            step=1000.0
        )

        home = st.selectbox(
            "Home Ownership",
            ["RENT", "OWN", "MORTGAGE", "OTHER"]
        )

        employment = st.number_input(
            "Employment Length (Years)",
            min_value=0.0,
            max_value=60.0,
            value=4.0,
            step=0.5
        )

        previous_default = st.selectbox(
            "Previous Default on File",
            ["N", "Y"],
            format_func=lambda x: "Yes" if x == "Y" else "No"
        )

    with col2:
        intent = st.selectbox(
            "Loan Purpose",
            [
                "PERSONAL",
                "EDUCATION",
                "MEDICAL",
                "VENTURE",
                "HOMEIMPROVEMENT",
                "DEBTCONSOLIDATION"
            ]
        )

        grade = st.selectbox(
            "Internally Assigned Loan Grade",
            ["A", "B", "C", "D", "E", "F", "G"]
        )

        amount = st.number_input(
            "Requested Loan Amount",
            min_value=1.0,
            value=15000.0,
            step=500.0
        )

        credit_history = st.number_input(
            "Credit History Length (Years)",
            min_value=0,
            max_value=50,
            value=5
        )

    submitted = st.form_submit_button(
        "Predict Default Risk",
        use_container_width=True
    )

# Make a prediction only when the form is submitted
if submitted:

    # Map letter grades to the encoded training values
    grade_value = ord(grade) - ord("A")

    # Calculate the loan-to-income ratio
    loan_percent_income = amount / income

    # Build one record with the exact training features
    applicant = pd.DataFrame([{
        "person_age": int(age),
        "person_income": float(income),
        "person_home_ownership": home,
        "person_emp_length": float(employment),
        "loan_intent": intent,
        "loan_grade": int(grade_value),
        "loan_amnt": float(amount),
        "loan_percent_income": float(loan_percent_income),
        "cb_person_default_on_file": previous_default,
        "cb_person_cred_hist_length": int(credit_history)
    }])

    # Calculate the estimated probability of default
    probability = float(
        model.predict_proba(applicant)[0, 1]
    )

    st.divider()
    st.header("Prediction Result")

    st.metric(
        "Estimated Probability of Default",
        f"{probability * 100:.2f}%"
    )

    st.progress(min(max(probability, 0.0), 1.0))

    # Apply the provisional referral threshold
    if probability >= THRESHOLD:
        st.warning("Refer to loan officer")
        st.write(
            "This application requires further "
            "assessment by a loan officer."
        )
    else:
        st.success("Below provisional referral threshold")
        st.write(
            "A loan officer must still complete "
            "the normal application review."
        )

    st.caption(
        "The 30% referral threshold is provisional. "
        "It was investigated in Task 5 using another "
        "classifier and must be validated separately "
        "for this Random Forest before real use."
    )

st.divider()
st.caption(
    "CSC 4110 | Kibabii University | Group G04"
)


Overwriting app/app.py


In [9]:

# Task 9.1(b): Create application requirements

import sklearn
import pandas
import joblib
import os

# Pin versions for model compatibility
requirements = (
    "streamlit\n"
    f"scikit-learn=={sklearn.__version__}\n"
    f"joblib=={joblib.__version__}\n"
    f"pandas=={pandas.__version__}\n"
)

os.makedirs("app", exist_ok=True)

with open("app/requirements.txt", "w") as f:
    f.write(requirements)

print("APP REQUIREMENTS CREATED")
print(requirements)
print("Saved: app/requirements.txt")


APP REQUIREMENTS CREATED
streamlit
scikit-learn==1.6.1
joblib==1.6.0
pandas==2.2.3

Saved: app/requirements.txt


In [10]:

# Install Streamlit if needed
%pip install -q streamlit


In [11]:

# Start the Streamlit application from Colab

import subprocess
import sys
import time

# Keep a log for troubleshooting
log_file = open("outputs/streamlit_log.txt", "w")

# Launch the app on port 8501
app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit",
        "run", "app/app.py",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Give the server a chance to start
time.sleep(5)

print("Application process running:",
      app_process.poll() is None)


Application process running: True


In [29]:

# Display the complete Streamlit application in Colab

from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    width="100%",
    height=1200
)


<IPython.core.display.Javascript object>

In [20]:

# Restart Streamlit for temporary Colab preview testing

import subprocess
import sys
import time
import requests

# Stop the previous Streamlit process
if "app_process" in globals() and app_process.poll() is None:
    app_process.terminate()
    try:
        app_process.wait(timeout=8)
    except subprocess.TimeoutExpired:
        app_process.kill()
        app_process.wait()

# Close the previous log file
if "log_file" in globals() and not log_file.closed:
    log_file.close()

# Start a fresh application process
log_file = open("outputs/streamlit_log.txt", "w")

app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit",
        "run", "app/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Check that the new server responds
time.sleep(4)

print("Streamlit running:", app_process.poll() is None)

try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Health status:", response.status_code)
except Exception as e:
    print("Connection error:", e)


Streamlit running: True
Connection error: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7dc3dc1ce0d0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [22]:

# Task 9.1: Automated Streamlit application test

from pathlib import Path
from streamlit.testing.v1 import AppTest
import joblib

print("=== MULIKA SACCO APPLICATION TEST ===")

# Test 1: Confirm required application files exist
required_files = [
    "app/app.py",
    "app/requirements.txt",
    "models/G04_model.joblib"
]

for path in required_files:
    assert Path(path).is_file(), f"Missing: {path}"
    print("PASS - File found:", path)

# Test 2: Confirm the model loads and predicts
model = joblib.load("models/G04_model.joblib")

prediction = model.predict(X_test.iloc[[0]])[0]
probability = model.predict_proba(
    X_test.iloc[[0]]
)[0, 1]

assert prediction in [0, 1]
assert 0 <= probability <= 1

print("\nPASS - Model predictions work")
print("Predicted class:", prediction)
print("Default probability:", round(probability * 100, 2), "%")

# Test 3: Launch the Streamlit app in test mode

# Use the full path to the Streamlit application
app_path = (Path.cwd() / "app" / "app.py").resolve()

print("Testing app:", app_path)
print("App file exists:", app_path.is_file())

# Launch the application in automated test mode
app_test = AppTest.from_file(
    str(app_path),
    default_timeout=120
).run()


assert len(app_test.exception) == 0, (
    "Application startup raised an exception: "
    + str(list(app_test.exception))
)

print("\nPASS - Streamlit application starts")

# Test 4: Find and click the prediction button
buttons = list(app_test.button)

submit_button = next(
    (
        button for button in buttons
        if button.label == "Predict Default Risk"
    ),
    None
)

assert submit_button is not None, (
    "Predict Default Risk button was not found"
)

app_test = submit_button.click().run()

assert len(app_test.exception) == 0, (
    "Prediction button raised an exception: "
    + str(list(app_test.exception))
)

print("PASS - Prediction button works")

# Test 5: Confirm a probability result is displayed
metric_labels = [
    metric.label for metric in app_test.metric
]

assert "Estimated Probability of Default" in metric_labels

print("PASS - Default probability displayed")

print("\nALL APPLICATION TESTS PASSED!")


=== MULIKA SACCO APPLICATION TEST ===
PASS - File found: app/app.py
PASS - File found: app/requirements.txt
PASS - File found: models/G04_model.joblib


2026-11-26 16:01:02.452 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.



PASS - Model predictions work
Predicted class: 0
Default probability: 11.0 %
Testing app: /content/drive/.shortcut-targets-by-id/1Hfur0Lf8z0gktOkV4oQdkhdGubaALOUm/CSC4110_G04/app/app.py
App file exists: True

PASS - Streamlit application starts
PASS - Prediction button works
PASS - Default probability displayed

ALL APPLICATION TESTS PASSED!


In [23]:

# Change the Streamlit application to a wide layout

from pathlib import Path

app_file = Path("app/app.py")

# Read the existing application code
code = app_file.read_text()

# Change the layout from centered to wide
code = code.replace(
    'layout="centered"',
    'layout="wide"'
)

# Save the updated application
app_file.write_text(code)

print("Application changed to wide layout successfully!")


Application changed to wide layout successfully!


In [30]:
# Improve the Mulika SACCO application layout

from pathlib import Path

app_file = Path("app/app.py")
code = app_file.read_text(encoding="utf-8")

# Locate the existing header section
start = code.find('st.title("🏦 Mulika SACCO")')
end = code.find("st.divider()", start)

if start == -1 or end == -1:
    raise ValueError("Application header could not be located.")

# Replace only the visual header and project information
new_header = '''
# Main application header
st.title("🏦 Mulika SACCO")
st.subheader("Loan Default Risk Screening")
st.caption("A machine learning decision-support tool for loan officers")

# Project information sidebar
with st.sidebar:
    st.header("Project Overview")
    st.write("**University:** Kibabii University")
    st.write("**Course:** CSC 4110")
    st.write("**Group:** G04")
    st.write("**Dataset:** Credit Risk Dataset")
    st.write("**Model:** Random Forest")
    st.write("**Referral Threshold:** 30% (Provisional)")

    st.divider()

    st.subheader("Project Team")
    st.write("Festus Ligeyo")
    st.write("Collins Kipkorir")
    st.write("Chirchir Collins")
    st.write("Peter Mbugua")
    st.write("Hartie Omuruon")

    st.divider()
    st.caption("Machine Learning Group Project")

# Important user guidance
st.info(
    "This application supports loan officers. "
    "It does not automatically approve or reject loans."
)

'''

# Preserve the remainder of the application
code = code[:start] + new_header + code[end:]

# Save the updated application
app_file.write_text(code, encoding="utf-8")

print("SUCCESS: Dashboard header and sidebar updated!")
print("Existing form and prediction logic preserved.")

SUCCESS: Dashboard header and sidebar updated!
Existing form and prediction logic preserved.


In [28]:

# Task 9.1: Hide project details in a collapsible sidebar

from pathlib import Path

app_file = Path("app/app.py")
code = app_file.read_text(encoding="utf-8")

# Find the current sidebar section
start = code.find("# Project information sidebar")
end = code.find("# Important user guidance", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate the sidebar section.")

# Replace the visible sidebar with collapsed project details
new_sidebar = '''
# Keep project information accessible but hidden by default
with st.sidebar:
    st.header("About This Project")

    with st.expander("Project Information", expanded=False):
        st.write("**University:** Kibabii University")
        st.write("**Course:** CSC 4110")
        st.write("**Group:** G04")
        st.write("**Dataset:** Credit Risk Dataset")
        st.write("**Model:** Random Forest")
        st.write("**Referral Threshold:** 30% (Provisional)")

    with st.expander("Group Members", expanded=False):
        st.write("Festus Ligeyo")
        st.write("Collins Kipkorir")
        st.write("Chirchir Collins")
        st.write("Peter Mbugua")
        st.write("Hartie Omuruon")

    st.caption("CSC 4110 Machine Learning Project")

'''

# Preserve the rest of the application
code = code[:start] + new_sidebar + code[end:]

# Close the entire sidebar when the app first opens
if "initial_sidebar_state=" not in code:
    code = code.replace(
        'layout="wide"',
        'layout="wide", initial_sidebar_state="collapsed"',
        1
    )

# Save the updated application
app_file.write_text(code, encoding="utf-8")

print("SUCCESS: Project information hidden by default!")
print("Applicant form and prediction logic preserved.")


SUCCESS: Project information hidden by default!
Applicant form and prediction logic preserved.


In [31]:

# Fix Streamlit app: Restore the missing model loader

from pathlib import Path
import ast

# Locate the existing application
app_file = Path("app/app.py")

# Read the current application code
code = app_file.read_text(encoding="utf-8")

# Define the missing model-loading section
model_loader = '''
# Load the trained Random Forest model
@st.cache_resource
def load_pipeline():
    return joblib.load(MODEL_PATH)

# Check that the saved model exists
if not MODEL_PATH.is_file():
    st.error("Trained model file not found.")
    st.stop()

# Make the model available for predictions
model = load_pipeline()

'''

# Insert the model loader before the applicant form
marker = 'st.header("Applicant Information")'

if "model = load_pipeline()" not in code:

    if marker not in code:
        raise ValueError(
            "Applicant Information section not found."
        )

    code = code.replace(
        marker,
        model_loader + marker,
        1
    )

    # Check Python syntax before saving
    ast.parse(code)

    # Save the repaired application
    app_file.write_text(code, encoding="utf-8")

    print("SUCCESS: Model loading restored!")

else:
    print("Model loader already exists.")

print("Application file:", app_file)
print("Model loader present:",
      "model = load_pipeline()" in code)


SUCCESS: Model loading restored!
Application file: app/app.py
Model loader present: True


In [32]:

# Task 9.1: Verify final application files

import os

required_files = [
    "models/G04_model.joblib",
    "app/app.py",
    "app/requirements.txt",
    "outputs/app_screenshot.png"
]

print("TASK 9.1 APPLICATION FILE CHECK")

for file in required_files:
    exists = os.path.isfile(file)
    print(file, ":", "FOUND" if exists else "MISSING")

print("\nVerification completed!")


TASK 9.1 APPLICATION FILE CHECK
models/G04_model.joblib : FOUND
app/app.py : FOUND
app/requirements.txt : FOUND
outputs/app_screenshot.png : MISSING

Verification completed!


## Part a) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [14]:
# a1. <what this cell does>


## Part b) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [15]:
# b1. <what this cell does>


## Part c) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [16]:
# c1. <what this cell does>


## Written answer

### Task 9.1: Written Answer — Model Packaging and Web Application

Our group began by acquiring the Credit Risk Dataset and cleaning it to remove duplicate records, handle missing values and correct unrealistic values.

We then explored and transformed the data before training regression and classification models to predict loan interest rates and loan defaults.

After comparing model performance, I retrained a Random Forest classifier within a complete preprocessing pipeline and saved it as `models/G04_model.joblib`.

I saved the entire pipeline instead of only the classifier so that new applicant information undergoes the same missing-value handling, categorical encoding and numerical scaling used during training.

I developed and tested a Streamlit application that accepts applicant information, calculates loan-default probability and supports loan-officer review rather than making automatic lending decisions.

The application will undergo further fairness checks, threshold validation and online deployment during the remaining group tasks.

## Reviews

### Review by <Name>
A check, an improvement or a question.

## Evidence checklist (tick before you submit)

- [ ] Runtime → Restart session and run all: no errors
- [ ] The basic check below shows no missing names or files
- [ ] The written answer is complete and quotes my numbers
- [ ] The self-check cell from Moodle passes
- [ ] File → Save, and File → Save a copy in GitHub

In [33]:
# Basic check: are the required variables and files there? (names and files only;
# the self-check cell on Moodle also tests the values)
import glob
REQUIRED = {
    "2.1": (["df_raw", "df_stage1", "outlier_col", "lower", "upper"], ["data/stage1_*.csv"]),
    "2.2": (["df_clean", "X_train", "X_test", "y_train", "y_test", "scaler"],
            ["data/clean_*.csv", "outputs/hist_target.png", "outputs/box_feature.png",
             "outputs/bar_class.png", "outputs/scatter_best.png", "outputs/corr_heatmap.png"]),
    "3.1": (["lr_simple", "lr_multi", "metrics_multi", "b0_gd", "b1_gd", "cost_history"],
            ["outputs/simple_fit.png", "outputs/actual_vs_pred.png", "outputs/gd_cost.png"]),
    "3.2": (["poly_results", "ridge_results", "lasso_results", "results_reg"],
            ["outputs/under_over_fit.png", "outputs/regression_comparison.png"]),
    "4":   (["y_train", "y_test", "log_reg", "cm", "results_clf"], ["outputs/confusion_matrix.png"]),
    "5":   (["pipe", "cv_scores", "grid", "test_score", "threshold_results"],
            ["outputs/learning_curve.png", "outputs/validation_curve.png"]),
    "6":   (["tree3", "ig_root", "depth_results", "results_trees", "importances"],
            ["outputs/tree_depth3.png", "outputs/depth_curve.png", "outputs/feature_importance.png"]),
    "7":   (["inertias", "sil_scores", "best_k", "pca", "dbscan_labels", "Q", "success_rates"],
            ["outputs/elbow.png", "outputs/pca_clusters.png", "outputs/success_rate.png"]),
    "8":   (["w_and", "b_and", "mlp_results", "best_mlp", "activation_results", "mlp_reg_rmse"], ["outputs/loss_curve.png"]),
    "9.1": (["final_pipe"], ["models/*_model.joblib", "app/app.py", "app/requirements.txt",
                             "outputs/app_screenshot.png"]),
    "9.2": (["fairness_table", "flagged"], ["outputs/fairness.csv", "outputs/fairness.png",
                                            "report/model_card.md"]),
    "9.3": ([], ["tests/smoke_test.py", "README.md"]),
}
names, files = REQUIRED[UNIT]
missing_names = [n for n in names if n not in globals()]
missing_files = [f for f in files if not glob.glob(f)]
print(f"Task {UNIT}")
print("Missing variables:", missing_names or "none")
print("Missing files:", missing_files or "none")
print("PASS" if not (missing_names or missing_files) else "FIX the items above before you submit")

Task 9.1
Missing variables: none
Missing files: ['outputs/app_screenshot.png']
FIX the items above before you submit


## Self-check

Paste the self-check cell for your task from its Moodle assignment below, then run it.

In [18]:
# Paste the self-check cell from Moodle here
